# Projection, shot surgery and working with a trained model

Once a reference model exists, new data can be handled in several ways:

| Situation | Method |
|---|---|
| cells from conditions the model knows | `get_latent` / `classify` directly, no training |
| new conditions, no time to train | `SCPOLI.shot_surgery` |
| new conditions, best quality | `SCPOLI.load_query_data` + `train` (tutorial 1) |

This tutorial also shows the scvi-tools functions the model inherits (normalized expression,
differential expression) and minification for sharing a trained model.

It uses the reference model saved by the first tutorial.

In [1]:
import logging
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scvi
from sklearn.metrics import accuracy_score

from scpolivi import SCPOLI

warnings.filterwarnings("ignore")
scvi.settings.seed = 0
scvi.settings.verbosity = logging.WARNING
sc.set_figure_params(figsize=(4.5, 4.5), frameon=False, dpi=80)

In [2]:
adata = sc.read("data/pancreas_sparse.h5ad")
model = SCPOLI.load("results/tutorials/reference_model")   # saved together with its AnnData
model

SCPOLI model: conditions ['study'] [7], cell types ['cell_type'] (13 prototypes), hidden [64], latent 10, 
recon_loss nb
Training status: Trained
Model's adata is minified?: False

## Projecting cells of known conditions

Cells from studies the model was trained on can be embedded and classified directly. They do
not need cell-type columns. Here we take inDrop1 cells, including the alpha cells the
reference never saw, drop their annotation and shuffle the gene order.

Genes are matched by name. `prepare_query_anndata` reorders them to the model's order and
zero-fills genes that are missing. Without that step, a different gene order raises an error
instead of silently producing wrong results.

In [3]:
new_cells = adata[adata.obs.study == "inDrop1"].copy()
truth = new_cells.obs.cell_type.astype(str).values
new_cells.obs = new_cells.obs[["study"]].copy()
new_cells = new_cells[:, np.random.default_rng(0).permutation(new_cells.n_vars)].copy()

try:
    model.get_latent(new_cells)
except ValueError as err:
    print("error:", err)

SCPOLI.prepare_query_anndata(new_cells, model)
res = model.classify(new_cells, use_mean=True, scale_uncertainties=True)["cell_type"]
seen = truth != "alpha"
print("accuracy on types known to the model:", round(accuracy_score(truth[seen], res["preds"][seen]), 3))
print("mean uncertainty  known types:", round(float(res["uncert"][seen].mean()), 3),
      " alpha:", round(float(res["uncert"][~seen].mean()), 3))

error: Genes are in a different order than in the model. Run `SCPOLI.prepare_query_anndata(adata, model)` first.
accuracy on types known to the model: 0.992
mean uncertainty  known types: 0.148  alpha: 0.561


Cells of a study the model has never seen cannot be projected directly, because there is no
embedding for that study:

In [4]:
query = adata[adata.obs.study.isin(["celseq", "celseq2"])].copy()
try:
    model.get_latent(query)
except ValueError as err:
    print("error:", err)

error: Conditions ['celseq', 'celseq2'] of 'study' are unknown to the model. Map them first with `SCPOLI.load_query_data` (or `shot_surgery`).


## Shot surgery: new conditions without training

`shot_surgery` gives each new study the embedding of the reference study under which its
cells are reconstructed best. It takes seconds, and label transfer is often almost as good
as with full query training.

In [5]:
shot_model, mapping = SCPOLI.shot_surgery(query, reference_model=model, labeled_indices=[])
mapping

{'celseq': 'smartseq2', 'celseq2': 'smartseq2'}

In [6]:
known = (query.obs.cell_type != "alpha").to_numpy()
pred = shot_model.classify(query, use_mean=True)["cell_type"]["preds"]
print("label transfer after shot surgery:", round(accuracy_score(query.obs.cell_type[known], pred[known]), 3))

label transfer after shot surgery: 0.979


## Inherited scvi-tools functions

SCPOLI is a regular scvi-tools model, so the decoder-based tools work as well. For example,
denoised expression of marker genes, decoded for one reference study
(`transform_batch` takes a combined condition):

In [7]:
reference = model.adata
markers = ["INS", "GCG", "SST", "PPY", "PRSS1", "KRT19"]
markers = [g for g in markers if g in reference.var_names]
expr = model.get_normalized_expression(gene_list=markers, transform_batch="inDrop1", library_size=1e4)
expr.groupby(reference.obs.cell_type.values).mean().round(1)

,GCG,INS,KRT19,PPY,PRSS1,SST
acinar,11.600000,29.400000,8.4,4.800000,121.5,35.200001
activated_stellate,13.600000,27.900000,0.8,5.000000,2.1,29.600000
beta,21.799999,189.600006,1.0,12.400000,5.4,78.400002
delta,27.000000,64.099998,4.2,13.500000,4.6,339.500000
ductal,14.000000,26.900000,42.5,5.000000,12.6,40.900002
endothelial,14.200000,41.200001,0.9,5.100000,3.7,45.900002
epsilon,25.600000,34.700001,4.4,74.000000,4.7,131.199997
gamma,23.799999,37.700001,1.8,249.699997,4.1,55.400002
macrophage,19.799999,40.099998,2.6,8.500000,3.3,34.599998
mast,20.200001,39.200001,2.4,9.500000,3.2,33.700001


The decoded expression follows the expected markers: INS is highest in beta cells, PPY in
gamma cells, PRSS1 in acinar cells and KRT19 in ductal cells.

Differential expression between two cell types of the reference.  reports
posterior log fold-changes (`lfc_mean`) and the probability that a gene is differentially
expressed (`proba_de`):

In [8]:
de = model.differential_expression(groupby="cell_type", group1="beta", group2="delta", mode="change")
cols = ["proba_de", "lfc_mean", "bayes_factor", "scale1", "scale2"]
top = pd.concat([de.nlargest(5, "lfc_mean")[cols], de.nsmallest(5, "lfc_mean")[cols]])
top.round(4)

DE...: 100%|██████████| 1/1 [00:00<00:00,  1.21it/s]


,proba_de,lfc_mean,bayes_factor,scale1,scale2
ADCYAP1,0.9960,3.6731,5.5175,0.0034,0.0003
MAFA,0.9900,3.4853,4.5951,0.0015,0.0001
G6PC2,0.9776,3.4287,3.7760,0.0035,0.0003
DLK1,0.9966,3.3633,5.6806,0.0030,0.0003
WSCD2,0.9812,2.9497,3.9549,0.0004,0.0001
HHEX,0.9986,-5.1427,6.5699,0.0000,0.0012
LY6H,0.9970,-5.0094,5.8061,0.0000,0.0010
LEPR,0.9992,-4.7390,7.1301,0.0001,0.0020
MS4A8,0.9972,-4.6165,5.8753,0.0000,0.0007
CBLN4,0.9986,-4.3705,6.5699,0.0000,0.0005


Known beta-cell genes (MAFA, G6PC2, ADCYAP1) are up in beta cells, and the delta-cell
transcription factor HHEX is up in delta cells.

## Minification

A trained model is often shared together with its data. `minify_adata` replaces the
expression matrix with the latent posterior (mean and variance per cell). Latents,
classification and normalized expression keep working; training does not.

In [9]:
z_full = model.get_latent(mean=True)
size_full = model.adata.X.data.nbytes / 1e6
model.minify_adata()
print(f"expression data: {size_full:.1f} MB -> {model.adata.X.nnz} stored values")
print("minified:", model.minified_data_type)
print("same latent:", np.allclose(model.get_latent(mean=True), z_full))

model.save("results/tutorials/reference_minified", overwrite=True, save_anndata=True)
small = SCPOLI.load("results/tutorials/reference_minified")
small.classify(use_mean=True)["cell_type"]["preds"][:5]

expression data: 19.2 MB -> 0 stored values
minified: latent_posterior_parameters
same latent: True


array(['gamma', 'gamma', 'gamma', 'gamma', 'gamma'], dtype='<U18')